# 20 · A low and a high QP model, each told the QP

Notebook 19's one told model beat the blind model in every QP bin, by more than 4 times a seed gap. The idea here: two
models, one for low QP and one for high, each still told the QP. Each half gets the whole body for its own job -
sharpening a clean frame, or deblocking a coded one - at the price of training on part of the data, shipping two sets of
weights, and a switch between them.

- **The switch is at QP 30**, which splits the patches about evenly. Notebook 18's specialist (QP 38 and up) had a
  quarter of the data and only tied the one model.
- **The training ranges overlap**: the low model trains on QP under 34, the high one on QP 26 and up, so each has seen
  frames on the far side of the switch. Each keeps its best epoch on validation patches from its own range.
- **The bar**: in each half, the split has to beat the one told model by clearly more than a seed gap. The one model's
  two seeds land at most 0.0004 SSIM-Y apart per bin (notebook 19), so about 0.001.
- **The switch itself**: how much the picture changes when a stream's QP crosses 30, against how much one QP step
  changes it in the one model. A large jump would show as a flicker when bandwidth moves around the switch.

The same body (24 channels, 2 blocks) and recipe for every model, two seeds each.

**Limits.** As in notebooks 18 and 19, the pairs are video only and P frames only.

Needs: notebook 17, part 5 (`qp_pairs/`), and notebook 19's two told runs in `checkpoints/qp_runs/`. Writes each
training of the low and high models there.

In [ ]:
"""Settings."""

import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, sources, training
from helpers.training import DEVICE

PAIRS_DIR = sources.ROOT / "upscale" / "qp_pairs"   # notebook 17, part 5
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"       # notebook 19's runs, and this notebook's
ONE_RUNS = {0: "told_qp", 1: "told_qp_seed1"}       # notebook 19's one told model, by seed

QP_EDGES = (0, 18, 26, 30, 34, 38, 42, 46)   # notebook 17's bins of a frame's QP
QP_MAX = 51                                  # the models are told QP / QP_MAX
SWITCH = 30                                  # the low model below it, the high one from it
OVERLAP = 4                                  # each model also trains this far past the switch
RANGES = {"low": (0, SWITCH + OVERLAP), "high": (SWITCH - OVERLAP, 99)}   # QP each model trains on
WIDTH, BLOCKS, STEM = 24, 2, 3               # notebook 8's body
BATCH_SIZE, LEARNING_RATE, MAX_EPOCHS = 32, 1e-3, 500
SEEDS = (0, 1)

training.seed_everything(SEEDS[0])

## 1. Data

Notebook 19's: every patch with its frame's QP, divided by 51. Below, how many patches each model trains and selects on.

In [ ]:
manifest = training.load_manifest(PAIRS_DIR)
training.describe(manifest)
SCALE = manifest["scale"]
trim = training.trimmer(manifest)


class WithQP(Dataset):
    """A split's pairs as ((LR, QP / QP_MAX), HR)."""

    def __init__(self, split, augment):
        self.pairs = patches.PatchPairs(PAIRS_DIR, split, manifest, augment=augment)
        self.qp = np.load(PAIRS_DIR / manifest["splits"][split]["qp"])
        self.condition = torch.from_numpy(self.qp / QP_MAX).float()[:, None, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr, hr = self.pairs[index]
        return (lr, self.condition[index]), hr


def batches(dataset, low=0, high=99, train=False):
    """A loader of `dataset`'s patches at QP from `low` up to `high`."""
    kept = Subset(dataset, np.flatnonzero((dataset.qp >= low) & (dataset.qp < high)))
    return DataLoader(kept, batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train)


train_set, val_set = WithQP("train", augment=True), WithQP("val", augment=False)
BINS = list(zip(QP_EDGES[:-1], QP_EDGES[1:]))
ROWS = [("all", 0, 99), *[(f"{low}-{high}", low, high) for low, high in BINS]]   # what every table is split by
print(f"{'patches':<8}{'all':>8}" + "".join(f"{f'QP {low}-{high}':>12}" for low, high in RANGES.values()))
for name, dataset in (("train", train_set), ("val", val_set)):
    print(f"{name:<8}{len(dataset):8}" + "".join(f"{((dataset.qp >= low) & (dataset.qp < high)).sum():12}"
                                                 for low, high in RANGES.values()))

## 2. Models

Every model is notebook 19's told body. `Split` is the pair as a client would run it: each frame through the low model
if its QP is under `SWITCH`, else through the high one.

In [ ]:
def build():
    return models.ToldLuma(conditions=1, width=WIDTH, blocks=BLOCKS, stem=STEM, scale=SCALE)


class Split(nn.Module):
    """A low and a high QP model as one: each patch through the one its QP picks."""

    def __init__(self, low, high):
        super().__init__()
        self.low, self.high = low, high

    def forward(self, x, condition):
        high = (condition * QP_MAX).round() >= SWITCH   # (N, 1, 1, 1)
        return torch.where(high, self.high(x, condition), self.low(x, condition))


def from_run(name):
    """A saved training's best epoch."""
    model = build().to(DEVICE)
    model.load_state_dict(torch.load(RUN_DIR / f"{name}.pt", map_location=DEVICE, weights_only=True)["best"]["state"])
    return model.eval()


assert build().halo <= manifest["halo"], "the model reaches further than the patches' halo"

## 3. Train

Notebook 19's recipe: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`, the epoch with the best SSIM-Y kept, until it
has converged; the seed sets the starting weights and then the batches. Each model trains and selects on its own range
only. The one told model is notebook 19's, loaded. Run the cell again after an interruption and it carries on; a
finished training just loads.

In [ ]:
def train(part, seed):
    """A fresh told model trained on `part`'s QP range, or its saved training loaded."""
    low, high = RANGES[part]
    torch.manual_seed(seed)
    model = build().to(DEVICE)
    torch.manual_seed(seed)
    name = f"told_{part}_qp{low}-{high}_seed{seed}"
    history, best = training.train(model, batches(train_set, low, high, train=True), batches(val_set, low, high), trim,
                                   MAX_EPOCHS, learning_rate=LEARNING_RATE, loss_fn=metrics.ssim_l1_loss,
                                   select="ssim", checkpoint=RUN_DIR / f"{name}.pt", verbose=0)
    print(f"{name:<28} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f} on its range")
    return model.eval()


halves = {(part, seed): train(part, seed) for seed in SEEDS for part in RANGES}
one = {seed: from_run(ONE_RUNS[seed]) for seed in SEEDS}
split = {seed: Split(halves["low", seed], halves["high", seed]) for seed in SEEDS}

## 4. Gain over bicubic, by QP

Each cell is SSIM-Y and luma dB over bicubic on that bin's validation patches: the one told model and the split, at
both seeds. The second table is the split's lead over the one model (the mean of two seeds each), and how far apart
each one's two seeds land.

In [ ]:
class Bicubic(nn.Module):
    def forward(self, x, condition=None):
        return models.bicubic(x, SCALE)


columns = {**{f"one s{seed}": model for seed, model in one.items()},
           **{f"split s{seed}": model for seed, model in split.items()}}
scores, counts = {}, {}   # (row, column) -> training.evaluate's scores; row -> patches
for label, low, high in ROWS:
    loader = batches(val_set, low, high)
    counts[label] = len(loader.dataset)
    scores[label, "bicubic"] = training.evaluate(Bicubic(), loader, trim)
    for name, model in columns.items():
        scores[label, name] = training.evaluate(model, loader, trim)


def gain(label, name, key):
    return scores[label, name][key] - scores[label, "bicubic"][key]


def mean_gain(label, kind, key):
    return np.mean([gain(label, f"{kind} s{seed}", key) for seed in SEEDS])


def seed_gap(label, kind, key):
    return abs(gain(label, f"{kind} s{SEEDS[0]}", key) - gain(label, f"{kind} s{SEEDS[1]}", key))


print(f"{'QP':>7}{'patches':>9}{'bicubic':>9}" + "".join(f"{name:>18}" for name in columns))
print(f"{'':>25}" + "".join(f"{'ssim-y / dB':>18}" for _ in columns))
for label, _, _ in ROWS:
    cells = [f"{gain(label, name, 'ssim'):+.4f} / {gain(label, name, 'y'):+.2f}" for name in columns]
    print(f"{label:>7}{counts[label]:9}{scores[label, 'bicubic']['ssim']:9.4f}" + "".join(f"{cell:>18}" for cell in cells))

print()
print(f"{'QP':>7}{'split - one':>20}{'one s0 - s1':>20}{'split s0 - s1':>20}")
for label, _, _ in ROWS:
    lead = f"{mean_gain(label, 'split', 'ssim') - mean_gain(label, 'one', 'ssim'):+.4f} / " \
           f"{mean_gain(label, 'split', 'y') - mean_gain(label, 'one', 'y'):+.2f}"
    rulers = [f"{seed_gap(label, kind, 'ssim'):.4f} / {seed_gap(label, kind, 'y'):.2f}" for kind in ("one", "split")]
    print(f"{label:>7}{lead:>20}" + "".join(f"{ruler:>20}" for ruler in rulers))

## 5. The switch

On the validation patches either side of the switch (QP 26 to 34), each frame through both halves and through the one
model told the QP one step apart:

- **split, low vs high**: the mean and the 99th percentile of the luma difference between the low and the high model's
  picture of the same frame, in 8-bit levels. This is what changes on screen when a stream's QP crosses 30.
- **one, QP vs QP + 1**: the same for the one model told the frame's QP and one more. This is what one QP step changes
  without a switch.

And each half's score on the far side of the switch, inside its overlap, against the other half on its own side.

In [ ]:
def luma_255(x):
    return metrics.rgb_to_yuv(trim(x).clamp(0, 1))[:, :1] * 255


@torch.no_grad()
def differences(first, second, low, high):
    """Per-pixel luma differences (8-bit levels) between two models' pictures of the patches at QP `low` to `high`.
    Each model is a function of (LR, condition)."""
    out = []
    for (lr, condition), _ in batches(val_set, low, high):
        lr, condition = lr.to(DEVICE), condition.to(DEVICE)
        out.append((luma_255(first(lr, condition)) - luma_255(second(lr, condition))).abs().flatten().cpu())
    return torch.cat(out)


low, high = SWITCH - OVERLAP, SWITCH + OVERLAP
print(f"luma change, QP {low}-{high} ({len(batches(val_set, low, high).dataset)} patches), 8-bit levels")
print(f"{'':<28}{'mean':>8}{'99th %':>8}")
for seed in SEEDS:
    pairs = {f"split s{seed}, low vs high": (halves["low", seed], halves["high", seed]),
             f"one s{seed}, QP vs QP + 1": (one[seed], lambda x, c, model=one[seed]: model(x, c + 1 / QP_MAX))}
    for name, (first, second) in pairs.items():
        change = differences(first, second, low, high)
        print(f"{name:<28}{change.mean():8.2f}{np.percentile(change.numpy(), 99):8.2f}")

print()
print(f"{'QP':>7}" + "".join(f"{f'low s{seed}':>18}{f'high s{seed}':>18}" for seed in SEEDS))
for label, low, high in ROWS:
    if low < SWITCH - OVERLAP or high > SWITCH + OVERLAP:
        continue
    loader, base = batches(val_set, low, high), scores[label, "bicubic"]
    cells = []
    for seed in SEEDS:
        for part in RANGES:
            s = training.evaluate(halves[part, seed], loader, trim)
            cells.append(f"{s['ssim'] - base['ssim']:+.4f} / {s['y'] - base['y']:+.2f}")
    print(f"{label:>7}" + "".join(f"{cell:>18}" for cell in cells))

## 6. What it says

**The split does not pay: one told model.** Over every patch the split is level with the one model (+0.0001 SSIM-Y,
+0.03 dB, inside both seed gaps). By half:

- **High QP (30 and up): level.** +0.0001 SSIM-Y in each bin from 34, and -0.02 to -0.04 dB. A model that only ever sees
  coded frames does no better there than the one that also sees clean ones: notebook 18's specialist said the same.
- **Low QP (under 26): a little better, under the bar.** +0.0009 SSIM-Y / +0.20 dB at QP 0-18 and +0.0004 / +0.10 at
  18-26. Both seeds agree, and that is 2 to 4 times the seed gaps, so it is likely real, but it is below the 0.001
  SSIM-Y set as the price of a second model.
- **At the switch: worse.** At QP 30-34 the split is -0.0004 SSIM-Y / -0.09 dB behind the one model. There the high
  model is ahead of the low one in SSIM-Y (+0.0354 against +0.0346) but behind it in dB (+1.34 against +1.45). At 26-30
  the low model beats the high one in both. Neither half is as good as the one model near the switch.

**The switch would show.** On the same frames at QP 26 to 34, the low and the high model's pictures differ by 1.1 luma
levels on average and by 12 at the 99th percentile. One QP step changes the one model's picture by 0.2 and 1.5. When a
stream's QP crosses 30, 1 % of pixels would jump by a dozen levels or more: a flicker every time the bandwidth
moves around the switch. The one model moves smoothly instead.

So the low end has a little more to give (up to +0.2 dB at QP 0-18), but a second model is not the way to take it: it
costs a second set of weights, a visible jump, and a loss near the switch. The one told model of notebook 19
(`checkpoints/upscale_qp.pt`) stays the model to carry on with.